# 04 — Delta hedging: what the theory promises and what you get

You sell an option at an implied vol and hedge it with shares. In continuous time
and perfect markets the hedge is riskless, and the PnL is the **gamma rent**:

    PnL = 1/2 * integral Gamma_t S_t^2 (sigma_realised^2 - sigma_implied^2) dt

Reality inserts three frictions:

1. **discrete rebalancing** — the PnL becomes a random variable, with standard
   deviation shrinking like `1/sqrt(rebalances)`;
2. **being wrong about vol** — sell 20%, realise 28%, and you lose steadily;
3. **transaction costs** — proportional to how often you trade, and to gamma.

This notebook runs the engine in `optlab.hedging.simulate` and checks each claim.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optlab.pricing.black_scholes import bs_price, bs_greeks
from optlab.pricing.binomial import binomial_price
from optlab.pricing.monte_carlo import mc_price

from optlab.hedging.attribution import decompose_pnl
from optlab.hedging.simulate import simulate_delta_hedge
from optlab.pricing.monte_carlo import simulate_gbm_paths

S0 = K = 100.0
T, R = 1.0, 0.04
IMPLIED = 0.20

## 1. Right vol, right hedge: zero expected PnL

Sell 20% vol, realise 20%. The hedge is self-financing and starts at zero value,
so the terminal PnL should average to zero — with a spread driven entirely by the
fact that we only rebalance daily. Note the standard error: this is a Monte Carlo
statement, not an exact one.

In [ ]:
matched = simulate_delta_hedge(S0, K, T, R, IMPLIED, IMPLIED, n_steps=252, n_paths=4000, seed=42)
print(matched.summary())

## 2. Volatility mismatch: the only thing that really matters

Sign the trade the other way in your head: long gamma when realised > implied.
The table below is the PnL of a *long* option position hedged with shares — so
when the market moves more than you paid for, you make money.

In [ ]:
rows = []
for realised in [0.12, 0.16, 0.20, 0.24, 0.28, 0.32]:
    res = simulate_delta_hedge(S0, K, T, R, IMPLIED, realised, n_steps=252, n_paths=4000, seed=42)
    rows.append({"realised": realised, "mean_pnl": res.mean, "std": res.std,
                 "95% CI half-width": 1.96 * res.stderr, "p05": res.quantiles()["p05"],
                 "p95": res.quantiles()["p95"]})
vol_table = pd.DataFrame(rows)
vol_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(vol_table["realised"] * 100, vol_table["mean_pnl"], "o-")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(IMPLIED * 100, color="grey", linestyle=":", label="vol sold (20%)")
ax.set_xlabel("realised vol (%)")
ax.set_ylabel("mean PnL per option")
ax.set_title("Sell 20% vol, hedge daily: PnL is a function of realised vol")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Rebalancing frequency: the cost of being discrete

Both the *spread* of PnL and the trading costs scale with how often you trade.
The error falls like `1/sqrt(n)`, so quadrupling your work only halves your risk —
the classic reason desks hedge less often than theory suggests.

In [ ]:
rows = []
for n_steps in [2, 5, 12, 26, 52, 252]:
    res = simulate_delta_hedge(S0, K, T, R, IMPLIED, IMPLIED, n_steps=n_steps, n_paths=4000, seed=42)
    costly = simulate_delta_hedge(S0, K, T, R, IMPLIED, IMPLIED, n_steps=n_steps, n_paths=4000,
                                  seed=42, transaction_cost_bps=10.0)
    rows.append({"rebalances": n_steps, "std_pnl": res.std,
                 "mean_pnl": res.mean,
                 "cost_at_10bps": costly.mean_cost,
                 "mean_pnl_net": costly.mean})
rebal = pd.DataFrame(rows)
rebal.round(4)

## 4. Attribution: theta, gamma, financing, residual

Decompose the hedged PnL step by step into `Theta*dt`, `1/2 Gamma dS^2`,
`Vega*dsigma`, the **financing** of the hedge `-r (V - Delta S) dt`, and a residual
(higher-order terms plus discretisation error).

The financing line is the one people forget. By the Black-Scholes PDE, theta already
contains the option's own variance cost, so theta looks like a big negative number.
But holding the hedge means borrowing `V - Delta*S`, which costs interest — and that
carries back roughly `r (V - Delta S)` per year. Drop the financing leg and theta and
gamma appear not to add up, which looks like a deep insight and is in fact a
bookkeeping bug.

If the residual is large relative to theta and gamma, the Taylor story is not
describing your book.

In [ ]:
paths = simulate_gbm_paths(S0, R, 0.28, T, steps=252, n_paths=4000, seed=42)
attribution = decompose_pnl(paths, K=K, T=T, r=R, sigma=IMPLIED)
attribution.summary().round(4)

In [ ]:
components = attribution.components
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(components["hedged_total"], bins=60, alpha=0.7, label="realised hedged PnL")
ax.hist(components["theta"] + components["gamma"], bins=60, alpha=0.5, label="theta + gamma")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("PnL per option")
ax.set_ylabel("paths")
ax.set_title("Taylor approximation vs realised hedged PnL (realised 28% vs implied 20%)")
ax.legend()
plt.tight_layout()
plt.show()

## Takeaway

* The hedge removes delta, not risk. What is left is a bet on realised vs implied vol.
* Hedging error is a real PnL line, not a rounding term — at daily frequency it is
  a few percent of notional per unit of vol mismatch.
* Costs grow with rebalancing frequency while risk shrinks like `1/sqrt(n)`: there
  is an optimal frequency, and it is not "as often as possible".

Reproduce these numbers from the command line with
`python scripts/run_hedging_experiment.py`.